# UK EV Charging Network Database (2025–2026)

**MSc Data Science – SQL Database Assignment**

This notebook generates a reproducible, entirely synthetic SQLite database representing a UK public electric-vehicle charging network. It creates related operational tables, applies integrity constraints, and demonstrates data-quality checks and example SQL analysis.

**Reproducibility:** the generator uses a fixed random seed (`20261105`). No third-party dataset or real personal data is used. Run the notebook from top to bottom. Keep `generate_ev_database.py` in the same folder as this notebook.

## 1. Database generation approach

The database separates reference entities (regions, stations, chargers, vehicles and tariffs) from transactional data (charging sessions and session events). This reduces repeated values and supports foreign-key validation. The `raw_session_imports` table simulates an imperfect incoming feed: missing fields and duplicate feed rows are intentionally inserted and labelled. The curated session table remains protected by primary and foreign keys.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

BASE_DIR = Path.cwd()
DB_PATH = BASE_DIR / "uk_ev_charging_2026.sqlite"
GENERATOR = BASE_DIR / "generate_ev_database.py"

assert GENERATOR.exists(), "Place generate_ev_database.py beside this notebook."

# Importing and calling the generator rebuilds the database deterministically.
from generate_ev_database import build_database
build_database(DB_PATH)
print("Created:", DB_PATH.resolve())

Created: /Users/nithin/Downloads/UK_EV_SQL_Assignment_Package/uk_ev_charging_2026.sqlite


## 2. Verify table sizes and relationships

The main transactional table contains 2,000 rows, exceeding the minimum 1,000-row requirement. Session events use a composite primary key (`session_id`, `event_sequence`). Tariffs use a three-column composite primary key.

In [2]:
with sqlite3.connect(DB_PATH) as con:
    table_names = ["regions", "stations", "chargers", "vehicles", "tariffs",
                   "charging_sessions", "session_events", "raw_session_imports"]
    counts = pd.DataFrame({
        "table": table_names,
        "rows": [pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", con).iloc[0,0]
                 for t in table_names]
    })
display(counts)

,table,rows
0,regions,8
1,stations,40
2,chargers,160
3,vehicles,350
4,tariffs,48
5,charging_sessions,2000
6,session_events,10000
7,raw_session_imports,2012


## 3. Inspect schema and data types

SQLite uses dynamic typing, so declared affinities and `CHECK` constraints are used to guide valid values. In the generated data, connector type and payment method are nominal; accessibility rating is ordinal; ambient temperature in Celsius is interval; and energy, duration and battery capacity are ratio measurements.

In [3]:
with sqlite3.connect(DB_PATH) as con:
    for table in table_names:
        print(f"\\n--- {table} ---")
        display(pd.read_sql_query(f"PRAGMA table_info({table})", con))
    print("Foreign key violations:")
    display(pd.read_sql_query("PRAGMA foreign_key_check", con))

\n--- regions ---


,cid,name,type,notnull,dflt_value,pk
0,0,region_id,INTEGER,0,None,1
1,1,region_name,TEXT,1,None,0
2,2,country,TEXT,1,None,0


\n--- stations ---


,cid,name,type,notnull,dflt_value,pk
0,0,station_id,INTEGER,0,None,1
1,1,region_id,INTEGER,1,None,0
2,2,station_name,TEXT,1,None,0
3,3,town,TEXT,1,None,0
4,4,postcode_area,TEXT,1,None,0
5,5,site_type,TEXT,1,None,0
6,6,opened_on,TEXT,1,None,0
7,7,accessibility_rating,TEXT,1,None,0
8,8,latitude,REAL,1,None,0
9,9,longitude,REAL,1,None,0


\n--- chargers ---


,cid,name,type,notnull,dflt_value,pk
0,0,charger_id,INTEGER,0,None,1
1,1,station_id,INTEGER,1,None,0
2,2,charger_label,TEXT,1,None,0
3,3,connector_type,TEXT,1,None,0
4,4,max_power_kw,REAL,1,None,0
5,5,charger_status,TEXT,1,None,0
6,6,installation_year,INTEGER,1,None,0


... (8 outputs left out)

\n--- raw_session_imports ---


,cid,name,type,notnull,dflt_value,pk
0,0,import_row_id,INTEGER,0,None,1
1,1,source_session_ref,TEXT,1,None,0
2,2,imported_at,TEXT,1,None,0
3,3,raw_energy_kwh,REAL,0,None,0
4,4,raw_payment_reference,TEXT,0,None,0
5,5,quality_flag,TEXT,1,None,0


Foreign key violations:


,table,rowid,parent,fkid


## 4. Data-quality checks

Missing model years are represented as SQL `NULL` where the synthetic vehicle record has an unknown year. Missing energy/payment-reference fields and repeated source references are deliberately present only in the raw staging table. This reflects a realistic ingestion problem while keeping the validated transactional data usable.

In [4]:
with sqlite3.connect(DB_PATH) as con:
    quality = pd.read_sql_query("""
        SELECT quality_flag, COUNT(*) AS records
        FROM raw_session_imports
        GROUP BY quality_flag
        ORDER BY quality_flag
    """, con)
    missing = pd.read_sql_query("""
        SELECT COUNT(*) AS missing_energy
        FROM raw_session_imports
        WHERE raw_energy_kwh IS NULL
    """, con)
    duplicates = pd.read_sql_query("""
        SELECT COUNT(*) AS duplicated_source_refs
        FROM (
            SELECT source_session_ref
            FROM raw_session_imports
            GROUP BY source_session_ref
            HAVING COUNT(*) > 1
        )
    """, con)
display(quality)
display(missing)
display(duplicates)

,quality_flag,records
0,Clean,1980
1,Duplicate feed row,12
2,Missing field,20


,missing_energy
0,21


,duplicated_source_refs
0,12


## 5. Example SQL analysis

These queries demonstrate joins, grouping, aggregation and time-based segmentation. They are illustrative analyses of the synthetic records, not claims about actual UK charging behaviour.

In [5]:
# Average energy delivered by connector type
with sqlite3.connect(DB_PATH) as con:
    connector_summary = pd.read_sql_query("""
        SELECT c.connector_type,
               COUNT(*) AS sessions,
               ROUND(AVG(s.energy_kwh), 2) AS mean_energy_kwh,
               ROUND(SUM(s.energy_kwh), 2) AS total_energy_kwh
        FROM charging_sessions AS s
        JOIN chargers AS c ON c.charger_id = s.charger_id
        GROUP BY c.connector_type
        ORDER BY total_energy_kwh DESC
    """, con)
display(connector_summary)

,connector_type,sessions,mean_energy_kwh,total_energy_kwh
0,CCS,1245,47.62,59287.35
1,Type 2,509,43.64,22210.49
2,CHAdeMO,246,50.85,12508.95


In [6]:
# Compare charging outcomes by site type and region
with sqlite3.connect(DB_PATH) as con:
    outcome_summary = pd.read_sql_query("""
        SELECT r.region_name, st.site_type, s.session_outcome,
               COUNT(*) AS session_count
        FROM charging_sessions AS s
        JOIN chargers AS c ON c.charger_id = s.charger_id
        JOIN stations AS st ON st.station_id = c.station_id
        JOIN regions AS r ON r.region_id = st.region_id
        GROUP BY r.region_name, st.site_type, s.session_outcome
        ORDER BY r.region_name, st.site_type, s.session_outcome
    """, con)
display(outcome_summary.head(20))

,region_name,site_type,session_outcome,session_count
0,East of England,Motorway,Completed,34
1,East of England,Motorway,Interrupted,6
2,East of England,Residential,Completed,84
3,East of England,Residential,Failed,4
4,East of England,Residential,Interrupted,10
5,East of England,Retail,Completed,49
6,East of England,Retail,Failed,2
7,East of England,Retail,Interrupted,4
8,East of England,Workplace,Completed,49
9,East of England,Workplace,Interrupted,4


In [7]:
# Monthly energy demand
with sqlite3.connect(DB_PATH) as con:
    monthly = pd.read_sql_query("""
        SELECT substr(started_at, 1, 7) AS month,
               COUNT(*) AS sessions,
               ROUND(SUM(energy_kwh), 1) AS energy_kwh
        FROM charging_sessions
        GROUP BY substr(started_at, 1, 7)
        ORDER BY month
    """, con)
display(monthly.head(12))

,month,sessions,energy_kwh
0,2025-01,79,3570.4
1,2025-02,88,3947.6
2,2025-03,101,4416.6
3,2025-04,101,4913.8
4,2025-05,90,4347.9
5,2025-06,83,3843.1
6,2025-07,96,4638.6
7,2025-08,93,4411.1
8,2025-09,87,3831.8
9,2025-10,89,3991.8


## 6. Design decisions and limitations

- **Normalisation:** regions, stations, chargers and vehicles are stored separately so that descriptive attributes are not repeated in every session.
- **Integrity:** foreign keys connect sessions to chargers and vehicles, events to sessions, and stations/tariffs to regions. `CHECK`, `UNIQUE`, `NOT NULL` and composite-key constraints restrict invalid combinations.
- **Synthetic realism:** plausible UK regions, EV manufacturers, connector standards, tariff periods, session durations and energy quantities are sampled from weighted distributions. The generated coordinates are illustrative and must not be used for navigation.
- **Data quality:** labelled missing and duplicate rows in the raw import table allow the effect of imperfect ingestion to be measured before cleaning.
- **Ethics:** pseudonymous vehicle-user labels are generated; no names, phone numbers, registration plates, precise home addresses or payment credentials are stored. Synthetic records avoid exposing real charging histories.
- **Limitations:** the database is a teaching simulation, not a statistically representative survey or a source of operational advice. The model simplifies real-world pricing, charger availability and battery behaviour.

## 7. Final validation

A successful run should show at least 2,000 curated charging sessions and no foreign-key violations. The database file created by this notebook is `uk_ev_charging_2026.sqlite`.

In [8]:
with sqlite3.connect(DB_PATH) as con:
    session_count = con.execute("SELECT COUNT(*) FROM charging_sessions").fetchone()[0]
    fk_issues = con.execute("PRAGMA foreign_key_check").fetchall()
    assert session_count >= 1000
    assert not fk_issues
    print(f"PASS: {session_count:,} charging sessions; no foreign-key violations.")

PASS: 2,000 charging sessions; no foreign-key violations.
